In [ ]:
import time
start_time = time.time()
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import spark_sdk as ss
import yaml
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, sum as sum_, count, max as max_, datediff, lit, min as min_, desc, when, unix_timestamp, row_number, struct, collect_list,\
to_date, countDistinct, exp, lit
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pytz

import spark_sdk as ss
ss.PySpark(yarn=False, num_executors=48, driver_memory='32g', executor_memory='32g',
           add_on_config0=('spark.port.maxRetries', '1000'),
           add_on_config1=("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"),
           add_on_config2=('spark.databricks.delta.retentionDurationCheck.enabled', 'false'),
           add_on_config3=('spark.sql.sources.partitionOverwriteMode', 'dynamic')
          )
spark = ss.PySpark().spark

from pyspark.sql.functions import to_date

full_dataset = spark.read.format('parquet').load('file:/home/baond12/projects/AI_Recommend_System_2025/BTL_HTTM/CS-recommendation-system/backend/full_dataset.parquet')\
.withColumn('event_date', to_date('TimePlaced'))

# Lấy danh sách 7 ngày cuối cùng có trong dữ liệu
max_date_row = full_dataset.agg(max_("event_date")).collect()[0][0]
# Tính toán ngày bắt đầu validation (cách max date 6 ngày)
val_date = max_date_row - timedelta(days=6)

print(f"Validation Date: {val_date}")

### Load glocal_k
user_item_recs_final_glocal = spark.read.format('parquet').load("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/user_item_recs_final_glocal.parquet")\
.where(f"d='{val_date.strftime('%Y-%m-%d')}'")

### Load glocal_k
item_metrics = spark.read.format('parquet').load("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/item_metrics.parquet")\
.where(f"d='{val_date.strftime('%Y-%m-%d')}'")

full_ = user_item_recs_final_glocal.withColumnRenamed('item_pred_score', 'user_item_feature')\
.join(item_metrics, ['ProductId'], 'left').fillna(0, subset=['final_score'])\
.withColumnRenamed('final_score', 'item_default_feature')

full_ = full_.withColumn('final_score', 0.7 * col('user_item_feature') +
        0.3 * col('item_default_feature'))\
.withColumn("rank", row_number().over(Window.partitionBy("Account_Id").orderBy(col("final_score").desc_nulls_last())))\
.select('Account_Id', 'ProductId', 'rank')

full_ = full_.withColumn('d', lit(val_date.strftime('%Y-%m-%d')))
full_.write.format('parquet')\
.mode('overwrite').option('overwriteSchema', 'true')\
.partitionBy(['d'])\
.save("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/combo_features_recoms_daily.parquet")